In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import ( 
    cross_val_score, GridSearchCV
)
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay,
)

In [2]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
 
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

In [5]:
DATA_DIR = "C:\\Users\\rashid\\Desktop\\customer_churn_prediction\\data\\processed"          # <-- update if your files live elsewhere
TARGET_COL = "Churn"       # <-- update to your actual target column name
 
X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")
y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv").squeeze("columns")
y_test = pd.read_csv(f"{DATA_DIR}/y_test.csv").squeeze("columns")
 
# Normalize target to 0/1 if it's still Yes/No text
if y_train.dtype == object:
    y_train = y_train.replace({"Yes": 1, "No": 0})
if y_test.dtype == object:
    y_test = y_test.replace({"Yes": 1, "No": 0})
y_train = y_train.astype(int)
y_test = y_test.astype(int)
 
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f} | Test churn rate: {y_test.mean():.3f}")
 

X_train shape: (5634, 30) | X_test shape: (1409, 30)
Train churn rate: 0.265 | Test churn rate: 0.265


In [6]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
 
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

In [24]:
# ---------------------------------------------------------------------
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),

    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
    ]),

    "Decision Tree": Pipeline([
        ("preprocess", preprocessor),
        ("model", DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)),
        # max_depth=4 keeps the tree small enough to actually read/present.
        # Drop the limit if you want max accuracy instead of a presentable tree.
    ]),
    
    "KNN": Pipeline([
        ("preprocess", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=15)),
        # KNN has no class_weight option — scaling (already in preprocessor)
        # matters a lot here since distance drives its predictions.
    ]),
}

In [18]:
def specificity_score(y_true, y_pred):
    """True negative rate — sklearn has no built-in for this."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)
 
results = []
 
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
 
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "Specificity": specificity_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_proba),
    }
    results.append(metrics)
 
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        if k != "Model":
            print(f"{k}: {v:.3f}")
 
    # Confusion matrix plot
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["No Churn", "Churn"], cmap="Blues"
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.savefig(f"confusion_matrix_{name.replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
    plt.close()


=== Logistic Regression ===
Accuracy: 0.740
Precision: 0.507
Recall: 0.786
Specificity: 0.724
F1-score: 0.616
ROC AUC: 0.841

=== Random Forest ===
Accuracy: 0.766
Precision: 0.550
Recall: 0.650
Specificity: 0.808
F1-score: 0.596
ROC AUC: 0.826


In [29]:
dt_pipe = models["Decision Tree"]
 
# Get readable feature names after one-hot encoding
ohe = dt_pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = (
    list(ohe.get_feature_names_out(categorical_features))
    if categorical_features
    else []
)
all_feature_names = numeric_features + cat_names
 
# Already fit inside the loop above, but re-fitting here makes this block
# safe to run standalone (e.g. if you copy just this section into a
# separate notebook cell for your presentation).
dt_pipe.fit(X_train, y_train)
dt_model = dt_pipe.named_steps["model"]
 
plt.figure(figsize=(20, 10))
plot_tree(
    dt_model,
    feature_names=all_feature_names,
    class_names=["No Churn", "Churn"],
    filled=True,
    rounded=True,
    fontsize=9,
    proportion=True,   # shows % of samples instead of raw counts, easier to explain
)
plt.title("Decision Tree — Churn Prediction (max depth = 4)")
plt.savefig("decision_tree.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved decision_tree.png — good for your 'Model(s) used' slide")
 

Saved decision_tree.png — good for your 'Model(s) used' slide


In [31]:
results_df = pd.DataFrame(results).set_index("Model")
print("\n=== Model comparison ===")
print(results_df.round(3))
 
best_model_name = results_df["ROC AUC"].idxmax()
print(f"\nBest model by ROC AUC: {best_model_name}")
 
# ROC curves for all models, overlaid for the presentation
fig, ax = plt.subplots()
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
plt.title("ROC Curve Comparison")
plt.savefig("roc_curve_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved roc_curve_comparison.png — good for your 'Model(s) used' slide")


=== Model comparison ===
                     Accuracy  Precision  Recall  Specificity  F1-score  \
Model                                                                     
Logistic Regression     0.740      0.507   0.786        0.724     0.616   
Random Forest           0.766      0.550   0.650        0.808     0.596   

                     ROC AUC  
Model                         
Logistic Regression    0.841  
Random Forest          0.826  

Best model by ROC AUC: Logistic Regression
Saved roc_curve_comparison.png — good for your 'Model(s) used' slide


In [36]:
cv_scores = cross_val_score(models[best_model_name], X_train, y_train, cv=5, scoring="roc_auc")
print("CV ROC-AUC scores:", cv_scores, "mean:", cv_scores.mean())

CV ROC-AUC scores: [0.86399916 0.85241061 0.85157045 0.83505404 0.82413972] mean: 0.845434796968228


In [38]:
param_grid = {"model__n_estimators": [100, 200, 300], "model__max_depth": [None, 8, 12]}
grid = GridSearchCV(models["Random Forest"], param_grid, scoring="roc_auc", cv=5)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_, "Best score:", grid.best_score_)

Best params: {'model__max_depth': 8, 'model__n_estimators': 100} Best score: 0.8455239501272146
